In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from scipy.io import loadmat
from tifffile import TiffFile
import cv2

In [2]:
sys.path.append(r'C:\Users\97254\Desktop\git\FluoroVision')

In [3]:
from src.post_process.fluorophore_intensity import FluorophoreIntensityEstimator
from src.utils.common_utils import xcycwh_to_x1y1x2y2

#### Functions

In [17]:
def plot_debug_steps(bead, peaks, peak_mask, interpolated_image,optimal_mask,intensity, cmap='gray'):
    fig, ax = plt.subplots(2, 2, figsize=(10, 5))
    im0 = ax[0, 0].imshow(bead, cmap=cmap)
    ax[0,0].set_title('Bead')
    chbr0 = plt.colorbar(im0, ax=ax[0, 0], fraction=0.01, pad=0.04)
    chbr0.set_label('Intensity')
    ax[0, 1].imshow(peak_mask, cmap=cmap)
    ax[0, 1].set_title('Peak Mask')
    for peak in peaks:
        ax[0, 0].plot(peak[1], peak[0], 'ro', markersize=5, markeredgewidth=1)
        ax[0, 1].plot(peak[1], peak[0], 'ro', markersize=5, markeredgewidth=1)
    im1 = ax[1, 0].imshow(interpolated_image, cmap=cmap)
    ax[1,0].set_title('Interpolated Image')
    chbr1 = plt.colorbar(im1, ax=ax[1, 0], fraction=0.01, pad=0.04)
    chbr1.set_label('Intensity')
    ax[1, 1].imshow(optimal_mask, cmap=cmap)
    ax[1, 1].set_title('Optimal Mask')
    fig.suptitle(f'Intensity: {intensity}')
    plt.tight_layout()
    plt.show()

#### Inputs

In [5]:
tif_path = r'c:\Users\97254\Desktop\Resources\Technion\exploratory_resaerach\fluorovision\data\A1\2025_02_05_A1.tif'

In [6]:
detection_path = r'c:\Users\97254\Desktop\Resources\Technion\exploratory_resaerach\fluorovision\data\A1\results\tracked_results.csv'

In [7]:
laser_map_path = r'C:\Users\97254\Desktop\git\FluoroVision\src\config\mapV2.mat'

In [8]:
df = pd.read_csv(detection_path)

In [ ]:
df.head()

In [10]:
fie = FluorophoreIntensityEstimator(map_path=laser_map_path, peak_radius=2, filter_size=20)

In [11]:
import numpy as np
from skimage.draw import polygon

def get_peak_mask_rectangle_gradient(image, peaks, radius):
    """
    Create a mask that covers rotated rectangular regions around each peak.
    The rectangle orientation is determined by the local image gradient.

    Parameters
    ----------
    image : np.ndarray
        2D image array.
    peaks : list of (int, int)
        List of (row, col) coordinates for each peak.

    Returns
    -------
    mask : np.ndarray (bool)
        Boolean mask with True for unmasked pixels and
        False for the masked (rectangular) regions.
    """
    # 1) Compute the image gradient (dy, dx)
    grad_row, grad_col = np.gradient(image)

    # Initialize mask as all True
    mask = np.ones_like(image, dtype=bool)

    # Size of half the rectangle side
    half_size = radius

    for (r, c) in peaks:
        # 2) Compute the orientation angle from the local gradient
        #    arctan2(dy, dx) gives the angle of the gradient vector
        angle = np.arctan2(grad_row[r, c], grad_col[r, c])

        # 3) Define the rectangle corners in local "x,y" space (center = (0,0))
        #    We'll define them in clockwise or counter-clockwise order:
        corners_x = np.array([-half_size,  half_size,  half_size, -half_size])
        corners_y = np.array([-half_size, -half_size,  half_size,  half_size])

        # 4) Rotate corners by 'angle' around (0,0)
        cosA, sinA = np.cos(angle), np.sin(angle)
        x_rot = corners_x * cosA - corners_y * sinA
        y_rot = corners_x * sinA + corners_y * cosA

        # 5) Shift corners so that (r,c) is the rectangle center
        #    (Remember: r ~ y, c ~ x)
        x_rot += c
        y_rot += r

        # 6) Rasterize the polygon defined by these corners
        #    into row/col indices
        rr, cc = polygon(y_rot, x_rot, shape=image.shape)

        # 7) Update the mask: set all pixels inside the polygon to False
        mask[rr, cc] = False

    return mask


In [ ]:
image_debug = 10

with TiffFile(tif_path) as tif:
    for i, page in enumerate(tif.pages):
        print(f'Processing frame {i+1}...')
        frame_df = df[df['frame'] == i+1]
        if frame_df.empty:
            continue
        boxes = frame_df.loc[:,['x_center', 'y_center', 'width', 'height']].to_numpy()
        frame = page.asarray()
        for box in boxes:
            x1, y1, x2, y2 = xcycwh_to_x1y1x2y2(box[0], box[1], box[2], box[3])
            bead = frame[y1:y2, x1:x2]
            peaks = fie.get_2d_peaks(bead)
            if len(peaks) < 3:
                continue
            dx,dy = np.gradient(bead)
            factor = fie.map[int(box[0]), int(box[1])]
            # peak radius method
            mask = fie.get_peak_mask_rectangle(bead, peaks)
            interpolated_image = fie.horizontal_axis_interpolation(bead, mask)
            _, _, closed_clusterd_array, _ = fie.kmean_cluster_2d_array(
            interpolated_image)
            optimal_mask = fie.get_optimal_rectangle(closed_clusterd_array)
            optimal_rectangle_intensity = np.sum(interpolated_image * optimal_mask)
            plot_debug_steps(bead, peaks, mask, interpolated_image, optimal_mask, optimal_rectangle_intensity/factor, 'viridis')
            # median fill method
            # median_interpolated_image = fie.horizontal_axis_median_fill(bead, mask)
            # _, _, closed_clusterd_array1, _ = fie.kmean_cluster_2d_array(
            # median_interpolated_image)
            # optimal_mask1 = fie.get_optimal_rectangle(closed_clusterd_array1)
            # optimal_rectangle_intensity1 = np.sum(median_interpolated_image * optimal_mask1)
            # print(f'Optimal rectangle intensity: {optimal_rectangle_intensity1/factor}')
            # plot_bead_and_mask(median_interpolated_image, optimal_mask1, 'viridis')
            # peak radius method with graident optimisation
            mask_1 = get_peak_mask_rectangle_gradient(bead, peaks, 2)
            interpolated_image1 = fie.horizontal_axis_interpolation(bead, mask_1)
            _, _, closed_clusterd_array1, _ = fie.kmean_cluster_2d_array(
            interpolated_image1)
            optimal_mask1 = fie.get_optimal_rectangle(closed_clusterd_array1)
            optimal_rectangle_intensity1 = np.sum(interpolated_image1 * optimal_mask1)
            plot_debug_steps(bead, peaks, mask_1, interpolated_image1, optimal_mask1, optimal_rectangle_intensity1/factor, 'viridis')
        if i == 20:
            break
